# ZGLMM159 — Cenário 1 — Centro 4058

**Regressão — centro dos 3 erros da validação anterior**
**Ambiente:** 🔴 **PRODUÇÃO (PRD)** · **Views:** `vw_zglmm159_c1`, `vw_zglmm159_m1`, `vw_zglmm159_m2`

**Objetivo:** Confirmar a correção dos 3 erros encontrados na migração (depósitos colapsados no M1, 434 materiais ausentes e linhas extras por tipo de avaliação) no mesmo centro em que foram achados.

---

## Como usar

1. Aperte **Run All**. Todas as células são **somente leitura** e independentes.
2. Nas células de **extração** (seções 4, 5 e 6), use *Download → Download full results* para gerar o CSV.
3. Extraia a **ZGLMM159 no SAP PRD** no **mesmo dia**, com o filtro abaixo.

### Filtro no SAP (tela de seleção da ZGLMM159)

| Campo | Valor |
|---|---|
| Material | `*` |
| Centro | **4058** |
| Depósito | _(vazio)_ |
| Org. Vendas | _(vazio)_ |
| Canal de Distribuição | _(vazio)_ |

### Volume esperado _(exploratória PRD)_

| View | Linhas | Chave |
|---|---|---|
| C1 | 194.485 | material + centro |
| M1 | 38.848 (30.301 materiais) | material + centro + depósito |
| M2 | 194.485 | material + centro |

### Regras de comparação _(da análise exploratória)_

| Regra | Detalhe |
|---|---|
| Colunas fora da comparação | Declaradas *sem fonte* e confirmadas 100% nulas — M1: `cod_custos_armazenagem`, `cod_ciclo_mrp`, `pct_refugo`, `vl_custos_fixos_por_pedido`, `qt_tempo_ciclo_trabalho` · M2: `ind_remessa_jit`, `ind_entrada_lotes`, `cod_calendario_planejamento`, `cod_perfil_cobertura`, `cod_margem_seguranca`, `cod_perfil_periodo`, `ind_utilizacao_quotizacao`, `ind_aap`, `qt_estoque_minimo_seguranca`, `pct_grau_atendimento`. Não aparecem nas extrações. |
| Material | Datalake **sem** zeros à esquerda — normalizar o SAP antes do cruzamento. |
| Data | `dt_inicio_validade_status` vem como **AAAAMMDD** — converter a data do SAP para o mesmo formato. |
| Itens para eliminação | `ind_marcado_exclusao = 'X'` está **incluído** nas views. Se o SAP não traz eliminados, a diferença esperada está na seção 7. |
| M1 sem depósito | A view M1 só traz linhas **com depósito**. Materiais do centro sem depósito estão na seção 9. |

## Seções

| # | Conteúdo |
|---|---|
| 1 | Volumetria das 3 views no centro |
| 2 | Chave única e linhas idênticas |
| 3 | Consistência entre C1, M1 e M2 |
| **4 · 5 · 6** | **Extração C1 · M1 · M2 (para cruzar com o SAP)** |
| 7 | Itens marcados para eliminação |
| 8 | Totais para conciliação |
| 9 | Materiais sem depósito (hipótese dos 434 ausentes) |
| 10 | Distribuições para conferência |
| 11 | Pontos de atenção da exploratória |
| 12 | Amostra para conferência na MM03 |
| 13 | Resumo do cenário |


## 1. Volumetria das 3 views no centro

Compare com o total de linhas de cada extração do SAP.

In [0]:
-- 1. VOLUMETRIA DO CENARIO
SELECT 'C1' AS view, COUNT(*) AS linhas, COUNT(DISTINCT cod_material) AS materiais, COUNT(DISTINCT cod_empresa) AS empresas,
       CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(cod_empresa))) AS lista_empresas
  FROM prd_procurement.corp_curated.vw_zglmm159_c1 WHERE cod_centro = '4058'
UNION ALL
SELECT 'M1', COUNT(*), COUNT(DISTINCT cod_material), COUNT(DISTINCT cod_empresa), CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(cod_empresa)))
  FROM prd_procurement.corp_curated.vw_zglmm159_m1 WHERE cod_centro = '4058'
UNION ALL
SELECT 'M2', COUNT(*), COUNT(DISTINCT cod_material), COUNT(DISTINCT cod_empresa), CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(cod_empresa)))
  FROM prd_procurement.corp_curated.vw_zglmm159_m2 WHERE cod_centro = '4058';

## 2. Chave única e linhas idênticas

Todos os valores de `duplicadas` e `identicas` devem ser **0**.

In [0]:
-- 2. CHAVE E CHECKSUM
SELECT 'C1' AS view, 'cod_material + cod_centro' AS chave, COUNT(*) AS linhas,
       COUNT(*) - COUNT(DISTINCT CONCAT_WS('|', cod_material, cod_centro)) AS duplicadas,
       COUNT(*) - COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`cod_empresa` AS STRING), ''), COALESCE(CAST(`cod_classe_avaliacao` AS STRING), ''), COALESCE(CAST(`tp_determinacao_preco` AS STRING), ''), COALESCE(CAST(`ind_controle_preco` AS STRING), ''), COALESCE(CAST(`qt_unidade_preco` AS STRING), ''), COALESCE(CAST(`vl_preco_medio_movel` AS STRING), ''), COALESCE(CAST(`ind_marcado_exclusao` AS STRING), '')))) AS identicas
  FROM prd_procurement.corp_curated.vw_zglmm159_c1 WHERE cod_centro = '4058'
UNION ALL
SELECT 'M1' AS view, 'cod_material + cod_centro + cod_deposito' AS chave, COUNT(*) AS linhas,
       COUNT(*) - COUNT(DISTINCT CONCAT_WS('|', cod_material, cod_centro, cod_deposito)) AS duplicadas,
       COUNT(*) - COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`cod_deposito` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`cod_empresa` AS STRING), ''), COALESCE(CAST(`tp_grupo_mrp` AS STRING), ''), COALESCE(CAST(`cod_regra_calculo_tamanho_lote` AS STRING), ''), COALESCE(CAST(`tp_mrp` AS STRING), ''), COALESCE(CAST(`cod_planejador_mrp` AS STRING), ''), COALESCE(CAST(`cod_horizonte_planejamento_fixo` AS STRING), ''), COALESCE(CAST(`cod_abc` AS STRING), ''), COALESCE(CAST(`tp_grupo_unidade_medida` AS STRING), ''), COALESCE(CAST(`cod_status_material_centro` AS STRING), ''), COALESCE(CAST(`cod_perfil_arredondamento` AS STRING), ''), COALESCE(CAST(`qt_tamanho_fixo_lote` AS STRING), ''), COALESCE(CAST(`qt_tamanho_maximo_lote` AS STRING), ''), COALESCE(CAST(`qt_tamanho_minimo_lote` AS STRING), ''), COALESCE(CAST(`qt_valor_arredondamento` AS STRING), ''), COALESCE(CAST(`qt_estoque_maximo` AS STRING), ''), COALESCE(CAST(`qt_ponto_reabastecimento` AS STRING), ''), COALESCE(CAST(`dt_inicio_validade_status` AS STRING), ''), COALESCE(CAST(`ind_marcado_exclusao` AS STRING), '')))) AS identicas
  FROM prd_procurement.corp_curated.vw_zglmm159_m1 WHERE cod_centro = '4058'
UNION ALL
SELECT 'M2' AS view, 'cod_material + cod_centro' AS chave, COUNT(*) AS linhas,
       COUNT(*) - COUNT(DISTINCT CONCAT_WS('|', cod_material, cod_centro)) AS duplicadas,
       COUNT(*) - COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`cod_empresa` AS STRING), ''), COALESCE(CAST(`tp_suprimento` AS STRING), ''), COALESCE(CAST(`tp_grupo_determinacao_estoque` AS STRING), ''), COALESCE(CAST(`cod_chave_horizonte` AS STRING), ''), COALESCE(CAST(`ind_coproduto` AS STRING), ''), COALESCE(CAST(`cod_deposito_suprimento_externo` AS STRING), ''), COALESCE(CAST(`cod_deposito_producao` AS STRING), ''), COALESCE(CAST(`ind_baixa_explosao` AS STRING), ''), COALESCE(CAST(`ind_material_granel` AS STRING), ''), COALESCE(CAST(`qt_margem_seguranca` AS STRING), ''), COALESCE(CAST(`tp_suprimento_especial` AS STRING), ''), COALESCE(CAST(`qt_tempo_producao_interna` AS STRING), ''), COALESCE(CAST(`qt_estoque_seguranca` AS STRING), ''), COALESCE(CAST(`qt_dias_prazo_entrega_previsto` AS STRING), ''), COALESCE(CAST(`qt_dias_processamento_entrada` AS STRING), ''), COALESCE(CAST(`ind_marcado_exclusao` AS STRING), '')))) AS identicas
  FROM prd_procurement.corp_curated.vw_zglmm159_m2 WHERE cod_centro = '4058';

## 3. Consistência entre C1, M1 e M2

- **C1 e M2** devem ter exatamente o mesmo conjunto de `material + centro` (na exploratória: 100% iguais).
- **M1** deve ser um subconjunto (só materiais com depósito).
- Empresa e indicador de eliminação não podem divergir entre as views.

In [0]:
-- 3. PRESENCA DE MATERIAL NAS 3 VIEWS
WITH c1 AS (SELECT DISTINCT cod_material FROM prd_procurement.corp_curated.vw_zglmm159_c1 WHERE cod_centro = '4058'),
     m1 AS (SELECT DISTINCT cod_material FROM prd_procurement.corp_curated.vw_zglmm159_m1 WHERE cod_centro = '4058'),
     m2 AS (SELECT DISTINCT cod_material FROM prd_procurement.corp_curated.vw_zglmm159_m2 WHERE cod_centro = '4058'),
     u  AS (SELECT * FROM c1 UNION SELECT * FROM m1 UNION SELECT * FROM m2)
SELECT CONCAT(CASE WHEN c1.cod_material IS NOT NULL THEN 'C1 ' ELSE '-- ' END,
              CASE WHEN m1.cod_material IS NOT NULL THEN 'M1 ' ELSE '-- ' END,
              CASE WHEN m2.cod_material IS NOT NULL THEN 'M2'  ELSE '--' END) AS presenca,
       COUNT(*) AS materiais,
       CASE WHEN c1.cod_material IS NULL OR m2.cod_material IS NULL THEN 'DIVERGENCIA C1 x M2'
            WHEN m1.cod_material IS NULL THEN 'esperado - sem deposito'
            ELSE 'ok' END AS veredito
FROM u
LEFT JOIN c1 ON u.cod_material = c1.cod_material
LEFT JOIN m1 ON u.cod_material = m1.cod_material
LEFT JOIN m2 ON u.cod_material = m2.cod_material
GROUP BY 1, 3 ORDER BY materiais DESC;

In [0]:
-- 3.1 EMPRESA E ELIMINACAO ENTRE AS VIEWS
WITH c1 AS (SELECT cod_material, cod_empresa, ind_marcado_exclusao FROM prd_procurement.corp_curated.vw_zglmm159_c1 WHERE cod_centro = '4058'),
     m1 AS (SELECT DISTINCT cod_material, cod_empresa, ind_marcado_exclusao FROM prd_procurement.corp_curated.vw_zglmm159_m1 WHERE cod_centro = '4058'),
     m2 AS (SELECT cod_material, cod_empresa, ind_marcado_exclusao FROM prd_procurement.corp_curated.vw_zglmm159_m2 WHERE cod_centro = '4058')
SELECT 'C1 x M2' AS par, COUNT(*) AS materiais_em_comum,
       COUNT_IF(COALESCE(c1.cod_empresa,'') <> COALESCE(m2.cod_empresa,'')) AS empresa_diverge,
       COUNT_IF(COALESCE(c1.ind_marcado_exclusao,'') <> COALESCE(m2.ind_marcado_exclusao,'')) AS eliminacao_diverge
  FROM c1 JOIN m2 ON c1.cod_material = m2.cod_material
UNION ALL
SELECT 'C1 x M1', COUNT(*),
       COUNT_IF(COALESCE(c1.cod_empresa,'') <> COALESCE(m1.cod_empresa,'')),
       COUNT_IF(COALESCE(c1.ind_marcado_exclusao,'') <> COALESCE(m1.ind_marcado_exclusao,''))
  FROM c1 JOIN m1 ON c1.cod_material = m1.cod_material;

## 4. Extração C1 — Contabilização 1

9 colunas (sem as declaradas *sem fonte*), ordenadas pela chave.
Use *Download full results* para exportar o CSV.

In [0]:
-- 4. EXTRACAO C1 - CENTRO 4058
SELECT `cod_material`,
       `cod_centro`,
       `cod_empresa`,
       `cod_classe_avaliacao`,
       `tp_determinacao_preco`,
       `ind_controle_preco`,
       `qt_unidade_preco`,
       `vl_preco_medio_movel`,
       `ind_marcado_exclusao`
FROM prd_procurement.corp_curated.vw_zglmm159_c1
WHERE cod_centro = '4058'
ORDER BY cod_material, cod_centro;

## 5. Extração M1 — MRP 1 (por depósito)

21 colunas (sem as declaradas *sem fonte*), ordenadas pela chave.
Use *Download full results* para exportar o CSV.

Chave: **material + centro + depósito**. Na exploratória, o centro tinha 38.848 linhas para 30.301 materiais.

In [0]:
-- 5. EXTRACAO M1 - CENTRO 4058
SELECT `cod_material`,
       `cod_deposito`,
       `cod_centro`,
       `cod_empresa`,
       `tp_grupo_mrp`,
       `cod_regra_calculo_tamanho_lote`,
       `tp_mrp`,
       `cod_planejador_mrp`,
       `cod_horizonte_planejamento_fixo`,
       `cod_abc`,
       `tp_grupo_unidade_medida`,
       `cod_status_material_centro`,
       `cod_perfil_arredondamento`,
       `qt_tamanho_fixo_lote`,
       `qt_tamanho_maximo_lote`,
       `qt_tamanho_minimo_lote`,
       `qt_valor_arredondamento`,
       `qt_estoque_maximo`,
       `qt_ponto_reabastecimento`,
       `dt_inicio_validade_status`,
       `ind_marcado_exclusao`
FROM prd_procurement.corp_curated.vw_zglmm159_m1
WHERE cod_centro = '4058'
ORDER BY cod_material, cod_centro, cod_deposito;

## 6. Extração M2 — MRP 2

18 colunas (sem as declaradas *sem fonte*), ordenadas pela chave.
Use *Download full results* para exportar o CSV.

In [0]:
-- 6. EXTRACAO M2 - CENTRO 4058
SELECT `cod_material`,
       `cod_centro`,
       `cod_empresa`,
       `tp_suprimento`,
       `tp_grupo_determinacao_estoque`,
       `cod_chave_horizonte`,
       `ind_coproduto`,
       `cod_deposito_suprimento_externo`,
       `cod_deposito_producao`,
       `ind_baixa_explosao`,
       `ind_material_granel`,
       `qt_margem_seguranca`,
       `tp_suprimento_especial`,
       `qt_tempo_producao_interna`,
       `qt_estoque_seguranca`,
       `qt_dias_prazo_entrega_previsto`,
       `qt_dias_processamento_entrada`,
       `ind_marcado_exclusao`
FROM prd_procurement.corp_curated.vw_zglmm159_m2
WHERE cod_centro = '4058'
ORDER BY cod_material, cod_centro;

## 7. Itens marcados para eliminação

Se o SAP **não** traz itens eliminados, a diferença de linhas entre SAP e Datalake deve ser igual à coluna `marcados_X`.

In [0]:
-- 7. ELIMINACAO POR VIEW
SELECT 'C1' AS view, COUNT(*) AS linhas, COUNT_IF(ind_marcado_exclusao = 'X') AS marcados_X,
       COUNT_IF(COALESCE(ind_marcado_exclusao,'') <> 'X') AS linhas_sem_eliminados
  FROM prd_procurement.corp_curated.vw_zglmm159_c1 WHERE cod_centro = '4058'
UNION ALL
SELECT 'M1', COUNT(*), COUNT_IF(ind_marcado_exclusao = 'X'), COUNT_IF(COALESCE(ind_marcado_exclusao,'') <> 'X')
  FROM prd_procurement.corp_curated.vw_zglmm159_m1 WHERE cod_centro = '4058'
UNION ALL
SELECT 'M2', COUNT(*), COUNT_IF(ind_marcado_exclusao = 'X'), COUNT_IF(COALESCE(ind_marcado_exclusao,'') <> 'X')
  FROM prd_procurement.corp_curated.vw_zglmm159_m2 WHERE cod_centro = '4058';

## 8. Totais para conciliação

Somas exatas (tipos `decimal`). Compare com a soma da mesma coluna na extração do SAP.

In [0]:
-- 8. TOTAIS
SELECT 'C1' AS view, 'qt_unidade_preco' AS coluna, CAST(SUM(`qt_unidade_preco`) AS STRING) AS total, COUNT(`qt_unidade_preco`) AS preenchidos, COUNT_IF(`qt_unidade_preco` <> 0) AS diferentes_de_zero FROM prd_procurement.corp_curated.vw_zglmm159_c1 WHERE cod_centro = '4058'
UNION ALL
SELECT 'C1' AS view, 'vl_preco_medio_movel' AS coluna, CAST(SUM(`vl_preco_medio_movel`) AS STRING) AS total, COUNT(`vl_preco_medio_movel`) AS preenchidos, COUNT_IF(`vl_preco_medio_movel` <> 0) AS diferentes_de_zero FROM prd_procurement.corp_curated.vw_zglmm159_c1 WHERE cod_centro = '4058'
UNION ALL
SELECT 'M1' AS view, 'qt_tamanho_fixo_lote' AS coluna, CAST(SUM(`qt_tamanho_fixo_lote`) AS STRING) AS total, COUNT(`qt_tamanho_fixo_lote`) AS preenchidos, COUNT_IF(`qt_tamanho_fixo_lote` <> 0) AS diferentes_de_zero FROM prd_procurement.corp_curated.vw_zglmm159_m1 WHERE cod_centro = '4058'
UNION ALL
SELECT 'M1' AS view, 'qt_tamanho_maximo_lote' AS coluna, CAST(SUM(`qt_tamanho_maximo_lote`) AS STRING) AS total, COUNT(`qt_tamanho_maximo_lote`) AS preenchidos, COUNT_IF(`qt_tamanho_maximo_lote` <> 0) AS diferentes_de_zero FROM prd_procurement.corp_curated.vw_zglmm159_m1 WHERE cod_centro = '4058'
UNION ALL
SELECT 'M1' AS view, 'qt_tamanho_minimo_lote' AS coluna, CAST(SUM(`qt_tamanho_minimo_lote`) AS STRING) AS total, COUNT(`qt_tamanho_minimo_lote`) AS preenchidos, COUNT_IF(`qt_tamanho_minimo_lote` <> 0) AS diferentes_de_zero FROM prd_procurement.corp_curated.vw_zglmm159_m1 WHERE cod_centro = '4058'
UNION ALL
SELECT 'M1' AS view, 'qt_valor_arredondamento' AS coluna, CAST(SUM(`qt_valor_arredondamento`) AS STRING) AS total, COUNT(`qt_valor_arredondamento`) AS preenchidos, COUNT_IF(`qt_valor_arredondamento` <> 0) AS diferentes_de_zero FROM prd_procurement.corp_curated.vw_zglmm159_m1 WHERE cod_centro = '4058'
UNION ALL
SELECT 'M1' AS view, 'qt_estoque_maximo' AS coluna, CAST(SUM(`qt_estoque_maximo`) AS STRING) AS total, COUNT(`qt_estoque_maximo`) AS preenchidos, COUNT_IF(`qt_estoque_maximo` <> 0) AS diferentes_de_zero FROM prd_procurement.corp_curated.vw_zglmm159_m1 WHERE cod_centro = '4058'
UNION ALL
SELECT 'M1' AS view, 'qt_ponto_reabastecimento' AS coluna, CAST(SUM(`qt_ponto_reabastecimento`) AS STRING) AS total, COUNT(`qt_ponto_reabastecimento`) AS preenchidos, COUNT_IF(`qt_ponto_reabastecimento` <> 0) AS diferentes_de_zero FROM prd_procurement.corp_curated.vw_zglmm159_m1 WHERE cod_centro = '4058'
UNION ALL
SELECT 'M2' AS view, 'qt_margem_seguranca' AS coluna, CAST(SUM(`qt_margem_seguranca`) AS STRING) AS total, COUNT(`qt_margem_seguranca`) AS preenchidos, COUNT_IF(`qt_margem_seguranca` <> 0) AS diferentes_de_zero FROM prd_procurement.corp_curated.vw_zglmm159_m2 WHERE cod_centro = '4058'
UNION ALL
SELECT 'M2' AS view, 'qt_tempo_producao_interna' AS coluna, CAST(SUM(`qt_tempo_producao_interna`) AS STRING) AS total, COUNT(`qt_tempo_producao_interna`) AS preenchidos, COUNT_IF(`qt_tempo_producao_interna` <> 0) AS diferentes_de_zero FROM prd_procurement.corp_curated.vw_zglmm159_m2 WHERE cod_centro = '4058'
UNION ALL
SELECT 'M2' AS view, 'qt_estoque_seguranca' AS coluna, CAST(SUM(`qt_estoque_seguranca`) AS STRING) AS total, COUNT(`qt_estoque_seguranca`) AS preenchidos, COUNT_IF(`qt_estoque_seguranca` <> 0) AS diferentes_de_zero FROM prd_procurement.corp_curated.vw_zglmm159_m2 WHERE cod_centro = '4058'
UNION ALL
SELECT 'M2' AS view, 'qt_dias_prazo_entrega_previsto' AS coluna, CAST(SUM(`qt_dias_prazo_entrega_previsto`) AS STRING) AS total, COUNT(`qt_dias_prazo_entrega_previsto`) AS preenchidos, COUNT_IF(`qt_dias_prazo_entrega_previsto` <> 0) AS diferentes_de_zero FROM prd_procurement.corp_curated.vw_zglmm159_m2 WHERE cod_centro = '4058'
UNION ALL
SELECT 'M2' AS view, 'qt_dias_processamento_entrada' AS coluna, CAST(SUM(`qt_dias_processamento_entrada`) AS STRING) AS total, COUNT(`qt_dias_processamento_entrada`) AS preenchidos, COUNT_IF(`qt_dias_processamento_entrada` <> 0) AS diferentes_de_zero FROM prd_procurement.corp_curated.vw_zglmm159_m2 WHERE cod_centro = '4058'
ORDER BY view, coluna;

## 9. Materiais sem depósito (hipótese dos 434 ausentes)

A view M1 exclui material + centro **sem depósito**. Na validação anterior, **434 materiais existiam no SAP e não no Datalake**.
Se o SAP M1 listar materiais sem depósito, eles estarão nesta lista — cruze os 434 com ela.

In [0]:
-- 9. MATERIAIS DO CENTRO SEM DEPOSITO (EM C1, FORA DO M1)
SELECT COUNT(*) AS materiais_sem_deposito,
       COUNT_IF(c.ind_marcado_exclusao = 'X') AS dos_quais_eliminados
FROM prd_procurement.corp_curated.vw_zglmm159_c1 c
LEFT ANTI JOIN (SELECT DISTINCT cod_material FROM prd_procurement.corp_curated.vw_zglmm159_m1 WHERE cod_centro = '4058') m ON c.cod_material = m.cod_material
WHERE c.cod_centro = '4058';

In [0]:
-- 9.1 LISTA PARA CRUZAR COM OS AUSENTES DO SAP
SELECT c.cod_material, c.cod_centro, c.cod_empresa, c.cod_classe_avaliacao, c.ind_marcado_exclusao
FROM prd_procurement.corp_curated.vw_zglmm159_c1 c
LEFT ANTI JOIN (SELECT DISTINCT cod_material FROM prd_procurement.corp_curated.vw_zglmm159_m1 WHERE cod_centro = '4058') m ON c.cod_material = m.cod_material
WHERE c.cod_centro = '4058'
ORDER BY c.cod_material;

## 10. Distribuições para conferência

Contagens por valor — devem bater com uma tabela dinâmica da extração do SAP.

In [0]:
-- 10. DISTRIBUICOES C1
SELECT 'ind_controle_preco' AS coluna, COALESCE(NULLIF(trim(CAST(`ind_controle_preco` AS STRING)), ''), '(vazio)') AS valor, COUNT(*) AS linhas FROM prd_procurement.corp_curated.vw_zglmm159_c1 WHERE cod_centro = '4058' GROUP BY 2
UNION ALL
SELECT 'tp_determinacao_preco' AS coluna, COALESCE(NULLIF(trim(CAST(`tp_determinacao_preco` AS STRING)), ''), '(vazio)') AS valor, COUNT(*) AS linhas FROM prd_procurement.corp_curated.vw_zglmm159_c1 WHERE cod_centro = '4058' GROUP BY 2
UNION ALL
SELECT 'cod_classe_avaliacao' AS coluna, COALESCE(NULLIF(trim(CAST(`cod_classe_avaliacao` AS STRING)), ''), '(vazio)') AS valor, COUNT(*) AS linhas FROM prd_procurement.corp_curated.vw_zglmm159_c1 WHERE cod_centro = '4058' GROUP BY 2
UNION ALL
SELECT 'ind_marcado_exclusao' AS coluna, COALESCE(NULLIF(trim(CAST(`ind_marcado_exclusao` AS STRING)), ''), '(vazio)') AS valor, COUNT(*) AS linhas FROM prd_procurement.corp_curated.vw_zglmm159_c1 WHERE cod_centro = '4058' GROUP BY 2
ORDER BY coluna, linhas DESC;

In [0]:
-- 10. DISTRIBUICOES M1
SELECT 'cod_deposito' AS coluna, COALESCE(NULLIF(trim(CAST(`cod_deposito` AS STRING)), ''), '(vazio)') AS valor, COUNT(*) AS linhas FROM prd_procurement.corp_curated.vw_zglmm159_m1 WHERE cod_centro = '4058' GROUP BY 2
UNION ALL
SELECT 'tp_mrp' AS coluna, COALESCE(NULLIF(trim(CAST(`tp_mrp` AS STRING)), ''), '(vazio)') AS valor, COUNT(*) AS linhas FROM prd_procurement.corp_curated.vw_zglmm159_m1 WHERE cod_centro = '4058' GROUP BY 2
UNION ALL
SELECT 'cod_abc' AS coluna, COALESCE(NULLIF(trim(CAST(`cod_abc` AS STRING)), ''), '(vazio)') AS valor, COUNT(*) AS linhas FROM prd_procurement.corp_curated.vw_zglmm159_m1 WHERE cod_centro = '4058' GROUP BY 2
UNION ALL
SELECT 'cod_regra_calculo_tamanho_lote' AS coluna, COALESCE(NULLIF(trim(CAST(`cod_regra_calculo_tamanho_lote` AS STRING)), ''), '(vazio)') AS valor, COUNT(*) AS linhas FROM prd_procurement.corp_curated.vw_zglmm159_m1 WHERE cod_centro = '4058' GROUP BY 2
UNION ALL
SELECT 'cod_planejador_mrp' AS coluna, COALESCE(NULLIF(trim(CAST(`cod_planejador_mrp` AS STRING)), ''), '(vazio)') AS valor, COUNT(*) AS linhas FROM prd_procurement.corp_curated.vw_zglmm159_m1 WHERE cod_centro = '4058' GROUP BY 2
UNION ALL
SELECT 'cod_status_material_centro' AS coluna, COALESCE(NULLIF(trim(CAST(`cod_status_material_centro` AS STRING)), ''), '(vazio)') AS valor, COUNT(*) AS linhas FROM prd_procurement.corp_curated.vw_zglmm159_m1 WHERE cod_centro = '4058' GROUP BY 2
UNION ALL
SELECT 'ind_marcado_exclusao' AS coluna, COALESCE(NULLIF(trim(CAST(`ind_marcado_exclusao` AS STRING)), ''), '(vazio)') AS valor, COUNT(*) AS linhas FROM prd_procurement.corp_curated.vw_zglmm159_m1 WHERE cod_centro = '4058' GROUP BY 2
ORDER BY coluna, linhas DESC;

In [0]:
-- 10. DISTRIBUICOES M2
SELECT 'tp_suprimento' AS coluna, COALESCE(NULLIF(trim(CAST(`tp_suprimento` AS STRING)), ''), '(vazio)') AS valor, COUNT(*) AS linhas FROM prd_procurement.corp_curated.vw_zglmm159_m2 WHERE cod_centro = '4058' GROUP BY 2
UNION ALL
SELECT 'tp_suprimento_especial' AS coluna, COALESCE(NULLIF(trim(CAST(`tp_suprimento_especial` AS STRING)), ''), '(vazio)') AS valor, COUNT(*) AS linhas FROM prd_procurement.corp_curated.vw_zglmm159_m2 WHERE cod_centro = '4058' GROUP BY 2
UNION ALL
SELECT 'cod_chave_horizonte' AS coluna, COALESCE(NULLIF(trim(CAST(`cod_chave_horizonte` AS STRING)), ''), '(vazio)') AS valor, COUNT(*) AS linhas FROM prd_procurement.corp_curated.vw_zglmm159_m2 WHERE cod_centro = '4058' GROUP BY 2
UNION ALL
SELECT 'cod_deposito_producao' AS coluna, COALESCE(NULLIF(trim(CAST(`cod_deposito_producao` AS STRING)), ''), '(vazio)') AS valor, COUNT(*) AS linhas FROM prd_procurement.corp_curated.vw_zglmm159_m2 WHERE cod_centro = '4058' GROUP BY 2
UNION ALL
SELECT 'cod_deposito_suprimento_externo' AS coluna, COALESCE(NULLIF(trim(CAST(`cod_deposito_suprimento_externo` AS STRING)), ''), '(vazio)') AS valor, COUNT(*) AS linhas FROM prd_procurement.corp_curated.vw_zglmm159_m2 WHERE cod_centro = '4058' GROUP BY 2
UNION ALL
SELECT 'ind_marcado_exclusao' AS coluna, COALESCE(NULLIF(trim(CAST(`ind_marcado_exclusao` AS STRING)), ''), '(vazio)') AS valor, COUNT(*) AS linhas FROM prd_procurement.corp_curated.vw_zglmm159_m2 WHERE cod_centro = '4058' GROUP BY 2
ORDER BY coluna, linhas DESC;

## 11. Pontos de atenção da exploratória

Colunas **sem declaração de "sem fonte"** que vieram **vazias ou sem valor útil em PRD** — confirmar no SAP se o campo também está vazio.
Se o SAP tiver valor, é **erro de carga**.

| View | Coluna | Situação em PRD (base inteira) |
|---|---|---|
| M1 | `tp_grupo_unidade_medida` | 100% vazio |
| M1 | `cod_perfil_arredondamento` | 100% vazio |
| M2 | `tp_grupo_determinacao_estoque` | 100% vazio |
| C1 | campos de avaliação | 8.398 linhas com tudo nulo (sem registro de avaliação) |
| M1 | vários campos MRP | 973 linhas com nulo |

In [0]:
-- 11. PREENCHIMENTO DOS PONTOS DE ATENCAO NO CENTRO
SELECT 'M1' AS view, 'tp_grupo_unidade_medida' AS coluna, COUNT(*) AS linhas,
       COUNT_IF(trim(COALESCE(tp_grupo_unidade_medida,'')) <> '') AS com_valor FROM prd_procurement.corp_curated.vw_zglmm159_m1 WHERE cod_centro = '4058'
UNION ALL
SELECT 'M1', 'cod_perfil_arredondamento', COUNT(*), COUNT_IF(trim(COALESCE(cod_perfil_arredondamento,'')) <> '') FROM prd_procurement.corp_curated.vw_zglmm159_m1 WHERE cod_centro = '4058'
UNION ALL
SELECT 'M2', 'tp_grupo_determinacao_estoque', COUNT(*), COUNT_IF(trim(COALESCE(tp_grupo_determinacao_estoque,'')) <> '') FROM prd_procurement.corp_curated.vw_zglmm159_m2 WHERE cod_centro = '4058'
UNION ALL
SELECT 'C1', 'avaliacao nula (ind_controle_preco)', COUNT(*), COUNT_IF(ind_controle_preco IS NULL) FROM prd_procurement.corp_curated.vw_zglmm159_c1 WHERE cod_centro = '4058'
UNION ALL
SELECT 'M1', 'MRP nulo (tp_grupo_mrp)', COUNT(*), COUNT_IF(tp_grupo_mrp IS NULL) FROM prd_procurement.corp_curated.vw_zglmm159_m1 WHERE cod_centro = '4058';

In [0]:
-- 11.1 MATERIAIS COM NULOS (CONFERIR NA MM03)
SELECT 'C1 - sem avaliacao' AS caso, cod_material, CAST(NULL AS STRING) AS cod_deposito
  FROM prd_procurement.corp_curated.vw_zglmm159_c1 WHERE cod_centro = '4058' AND ind_controle_preco IS NULL
UNION ALL
SELECT 'M1 - MRP nulo', cod_material, cod_deposito
  FROM prd_procurement.corp_curated.vw_zglmm159_m1 WHERE cod_centro = '4058' AND tp_grupo_mrp IS NULL
ORDER BY caso, cod_material
LIMIT 200;

## 12. Amostra para conferência na MM03

20 materiais fixos (sempre os mesmos, sorteio por hash). Um registro por material + depósito, com os campos principais das 3 views lado a lado.

In [0]:
-- 12. AMOSTRA FIXA
WITH amostra AS (
  SELECT cod_material FROM prd_procurement.corp_curated.vw_zglmm159_m1 WHERE cod_centro = '4058'
  GROUP BY cod_material ORDER BY md5(cod_material) LIMIT 20)
SELECT c1.cod_material, c1.cod_centro, m1.cod_deposito,
       c1.cod_classe_avaliacao, c1.ind_controle_preco, c1.vl_preco_medio_movel, c1.qt_unidade_preco,
       m1.tp_mrp, m1.cod_planejador_mrp, m1.cod_regra_calculo_tamanho_lote, m1.cod_abc,
       m1.qt_estoque_maximo, m1.qt_ponto_reabastecimento,
       m2.tp_suprimento, m2.qt_dias_prazo_entrega_previsto, m2.qt_dias_processamento_entrada, m2.qt_estoque_seguranca
FROM amostra a
JOIN prd_procurement.corp_curated.vw_zglmm159_c1 c1 ON c1.cod_material = a.cod_material AND c1.cod_centro = '4058'
JOIN prd_procurement.corp_curated.vw_zglmm159_m2 m2 ON m2.cod_material = a.cod_material AND m2.cod_centro = '4058'
JOIN prd_procurement.corp_curated.vw_zglmm159_m1 m1 ON m1.cod_material = a.cod_material AND m1.cod_centro = '4058'
ORDER BY c1.cod_material, m1.cod_deposito;

## 13. Resumo do cenário

**Copie esta saída** e envie junto com as extrações do SAP.

In [0]:
-- 13. RESUMO
WITH c1 AS (SELECT * FROM prd_procurement.corp_curated.vw_zglmm159_c1 WHERE cod_centro = '4058'),
     m1 AS (SELECT * FROM prd_procurement.corp_curated.vw_zglmm159_m1 WHERE cod_centro = '4058'),
     m2 AS (SELECT * FROM prd_procurement.corp_curated.vw_zglmm159_m2 WHERE cod_centro = '4058')
SELECT 'IDENTIFICACAO' AS bloco, 'cenario' AS item, 'ZGLMM159 - Cenario 1 - Centro 4058' AS valor
UNION ALL SELECT 'IDENTIFICACAO', 'ambiente', 'PRD'
UNION ALL SELECT 'IDENTIFICACAO', 'data_consulta', CAST(current_timestamp() AS STRING)
UNION ALL SELECT 'VOLUME', 'C1 linhas', format_number((SELECT COUNT(*) FROM c1), 0)
UNION ALL SELECT 'VOLUME', 'M1 linhas', format_number((SELECT COUNT(*) FROM m1), 0)
UNION ALL SELECT 'VOLUME', 'M1 materiais', format_number((SELECT COUNT(DISTINCT cod_material) FROM m1), 0)
UNION ALL SELECT 'VOLUME', 'M2 linhas', format_number((SELECT COUNT(*) FROM m2), 0)
UNION ALL SELECT 'CHAVE', 'C1 duplicadas', CAST((SELECT COUNT(*) - COUNT(DISTINCT cod_material) FROM c1) AS STRING)
UNION ALL SELECT 'CHAVE', 'M1 duplicadas', CAST((SELECT COUNT(*) - COUNT(DISTINCT CONCAT_WS('|', cod_material, cod_deposito)) FROM m1) AS STRING)
UNION ALL SELECT 'CHAVE', 'M2 duplicadas', CAST((SELECT COUNT(*) - COUNT(DISTINCT cod_material) FROM m2) AS STRING)
UNION ALL SELECT 'M1', 'linhas alem de 1 por material (depositos extras)',
       CAST((SELECT COUNT(*) - COUNT(DISTINCT cod_material) FROM m1) AS STRING)
UNION ALL SELECT 'M1', 'materiais sem deposito (em C1, fora do M1)',
       CAST((SELECT COUNT(*) FROM c1 LEFT ANTI JOIN (SELECT DISTINCT cod_material FROM m1) x ON c1.cod_material = x.cod_material) AS STRING)
UNION ALL SELECT 'ELIMINACAO', 'C1 marcados X', CAST((SELECT COUNT_IF(ind_marcado_exclusao = 'X') FROM c1) AS STRING)
UNION ALL SELECT 'ELIMINACAO', 'M1 marcados X', CAST((SELECT COUNT_IF(ind_marcado_exclusao = 'X') FROM m1) AS STRING)
UNION ALL SELECT 'ELIMINACAO', 'M2 marcados X', CAST((SELECT COUNT_IF(ind_marcado_exclusao = 'X') FROM m2) AS STRING)
ORDER BY bloco, item;

---

## Checklist de comparação com o SAP

- [ ] SAP extraído no mesmo dia, centro **4058**, demais campos vazios
- [ ] Material do SAP sem zeros à esquerda; data em AAAAMMDD
- [ ] Linhas por view batem (seção 1), descontando eliminados se for o caso (seção 7)
- [ ] M1: uma linha por material + depósito, sem colapso (seções 2 e 13)
- [ ] C1: uma linha por material, sem linhas extras por tipo de avaliação (seção 2)
- [ ] Ausentes no Datalake cruzados com a lista da seção 9.1
- [ ] Totais das colunas numéricas batem (seção 8)
- [ ] Pontos de atenção confirmados no SAP (seção 11)
- [ ] Amostra conferida na MM03 (seção 12)
